In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/movielends-25m-dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd, numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import os, glob

BASE = "/kaggle/input/movielens-25m-recsys/outputs/parquet"
parts = sorted(glob.glob(f"{BASE}/candidate_parts/candidates_part_*.parquet"))

candidates = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
item_features = pd.read_parquet(f"{BASE}/item_features.parquet")
user_genre    = pd.read_parquet(f"{BASE}/user_genre.parquet")
valid_label  = pd.read_parquet(f"{BASE}/valid_label.parquet")
test_label   = pd.read_parquet(f"{BASE}/test_label.parquet")

In [ ]:
user_genre = user_genre.reset_index()
user_genre = user_genre.rename(columns={"index":"u"})
user_genre.head()

In [ ]:
# join features
df = (
    candidates
    .merge(item_features, on="i", how="left")
    .merge(user_genre,    on="u", how="left")
)
df.head()

In [ ]:
df = df.rename(
    columns={c: c.replace("_y", "_user").replace("_x", "")
             for c in df.columns if c.endswith(("_x", "_y"))}
)

In [ ]:
# genre similarity
genre_cols = [c for c in item_features.columns if c.startswith("genre_")]
ug = df[[c+"_user" for c in genre_cols]].to_numpy()
ig = df[genre_cols].to_numpy()
df["genre_sim"] = (ug * ig).sum(axis=1)

In [ ]:
valid_label.head()

## Attach Labels to Valid and Test set for training

In [ ]:

valid_df = df[df["u"].isin(valid_label["u"])].merge(valid_label, on=["u","i"], how="left")
test_df  = df[df["u"].isin(test_label["u"])].merge(test_label,  on=["u","i"], how="left")

valid_df["label"] = (valid_df["rating"]>=4).astype(np.int8)
test_df["label"]  = (test_df["rating"]>=4).astype(np.int8)

## LightGBM

In [ ]:
features = ["mf_score","item_count","item_mean","item_recency01","genre_sim"]
train_data = lgb.Dataset(valid_df[features], label=valid_df["label"])
params = {"objective":"binary","metric":"auc","verbosity":-1}
gbm = lgb.train(params, train_data, num_boost_round=100)

## Evaluation

In [ ]:
def precision_at_k(df, k=10):
    out = []
    for u, g in df.groupby("u"):
        g = g.sort_values("pred", ascending=False)[:k]
        rel = g["label"].to_numpy()
        out.append(rel.mean())
    return np.mean(out)

valid_df["pred"] = gbm.predict(valid_df[features])
p10 = precision_at_k(valid_df, k=10)
print(f"Precision@10 = {p10:.4f}")

lgb.plot_importance(gbm, importance_type="gain")
plt.tight_layout()
plt.show()